# 06. Formatting Retrieved Recipes

The goal of this notebook is to document how we **format retrieved recipes**
from our cleaned dataset into the compact dictionaries used by:

- the RAG prompt builder, and
- the backend `/search_recipes` endpoint.

The final format of a retrieved recipe is a dict containing:

- `recipe_id`
- `title`
- `ingredients_list` (list or short string)
- `steps_list` (list or short string)
- `calories`, `fat`, `carbs`, `protein`
- (optionally) raw text fields for debugging

This transformation happens in `rag_pipeline/search.py`, but here we
demonstrate it with concrete examples.

In [7]:
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project root added: ", project_root)

Project root added:  /Users/biancaleoveanu/CookMate-Recipe-Generator


In [8]:
import pandas as pd
from rag_pipeline.search import search_recipes

## 1. Run a search and inspect a single result

In [10]:
from rag_pipeline.query_builder import build_query

query = build_query(
    ingredients=["chicken", "rice", "onion"],
    diet=None,
    cuisine=None
)

results = search_recipes(
    query,
    k=3,
)

len(results)

3

In [11]:
first = results[0]
first

{'recipe_id': 356574,
 'title': 'Best Rice Ever',
 'ingredients_list': ['yellow onion',
  'green bell pepper',
  'eggs',
  'Minute Rice',
  'chicken broth',
  'soya sauce',
  'salt',
  'pepper',
  'olive oil'],
 'ingredients_structured': [{'ingredient': 'yellow onion', 'quantity': '1'},
  {'ingredient': 'green bell pepper', 'quantity': '1'},
  {'ingredient': 'eggs', 'quantity': '1'},
  {'ingredient': 'Minute Rice', 'quantity': '2'},
  {'ingredient': 'chicken broth', 'quantity': '2'},
  {'ingredient': 'soya sauce', 'quantity': '2'},
  {'ingredient': 'salt', 'quantity': None},
  {'ingredient': 'pepper', 'quantity': None},
  {'ingredient': 'olive oil', 'quantity': None}],
 'steps_list': ['bring chicken broth to a boil, remove from heat. add minute rice and let stand for 10-15 mins, until water is absorbed, fluff with fork.',
  'meanwhile in a non stick skillet,  sautee onions and green pepper with a pinch of salt and pepper in 1 tbsp of olive oil until lightly browned. add mushrooms and s

We can inspect keys to understand the structure.

In [12]:
list(first.keys())

['recipe_id',
 'title',
 'ingredients_list',
 'ingredients_structured',
 'steps_list',
 'calories',
 'fat',
 'carbs',
 'protein']

## 2. Convert to a pandas DataFrame (for inspection only)

In [13]:
df_results = pd.DataFrame(results)
df_results.head()

,recipe_id,title,ingredients_list,ingredients_structured,steps_list,calories,fat,carbs,protein
0,356574,Best Rice Ever,"[yellow onion, green bell pepper, eggs, Minute...","[{'ingredient': 'yellow onion', 'quantity': '1...","[bring chicken broth to a boil, remove from he...",251.3,3.7,43.0,10.1
1,237250,Onion Rice,"[long-grain white rice, beef broth, vegetable ...","[{'ingredient': 'long-grain white rice', 'quan...","[In a medium saucepan over medium-high heat, c...",251.1,6.9,39.8,6.4
2,270606,Rice....oh My!,"[rice, red onion, olive oil, olive oil, low so...","[{'ingredient': 'rice', 'quantity': '1 1/2'}, ...","[Cut onions and pepper in small dice., Heat oi...",339.3,4.9,64.4,8.9


## 3. Explaining the formatting step

In `rag_pipeline/search.py`, for each retrieved index we build a dict like:

```python
{
    "recipe_id": int(row["recipe_id"]),
    "title": row["title"],
    "ingredients_list": row["ingredients_list"],
    "steps_list": row["steps_list"],
    "calories": float(row["Calories"]),
    "fat": float(row["FatContent"]),
    "carbs": float(row["CarbohydrateContent"]),
    "protein": float(row["ProteinContent"]),
}

In [14]:
def inspect_types(r: dict):
    for key, value in r.items():
        print(f"{key:16} ->", type(value), " | example:", str(value)[:50])

inspect_types(first)

recipe_id        -> <class 'int'>  | example: 356574
title            -> <class 'str'>  | example: Best Rice Ever
ingredients_list -> <class 'list'>  | example: ['yellow onion', 'green bell pepper', 'eggs', 'Min
ingredients_structured -> <class 'list'>  | example: [{'ingredient': 'yellow onion', 'quantity': '1'}, 
steps_list       -> <class 'list'>  | example: ['bring chicken broth to a boil, remove from heat.
calories         -> <class 'float'>  | example: 251.3
fat              -> <class 'float'>  | example: 3.7
carbs            -> <class 'float'>  | example: 43.0
protein          -> <class 'float'>  | example: 10.1


From here, the formatted recipes are passed to:

- `build_rag_prompt(user, retrieved)` (see notebook 07),
- the backend `/search_recipes` endpoint (used by the Streamlit frontend).

This concludes the formatting documentation.